In [2]:
!pip install -q transformers datasets accelerate

In [3]:


from __future__ import annotations

import argparse
import ast
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


DEFAULT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DEFAULT_SYSTEM_PROMPT = "You are an expert Python competitive programmer."


@dataclass
class Task:
    task_id: str
    prompt: str
    test: str
    entry_point: str


@dataclass
class TokenTrace:
    position: int
    token_id: int
    token_text: str
    entropy: float
    margin: float
    top1_id: int
    top1_text: str
    top2_id: int
    top2_text: str


@dataclass
class Generation:
    token_ids: list[int]
    code: str
    trace: list[TokenTrace]
    latency_s: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model-name", default=DEFAULT_MODEL)
    parser.add_argument("--output-dir", default="/kaggle/working/top2_counterfactual_pilot")
    parser.add_argument("--task-ids", default="", help="Comma-separated HumanEval task IDs.")
    parser.add_argument("--num-tasks", type=int, default=10, help="Used only when --task-ids is empty.")
    parser.add_argument("--max-new-tokens", type=int, default=256)
    parser.add_argument("--candidates-per-task", type=int, default=3)
    parser.add_argument("--controls-per-task", type=int, default=3)
    parser.add_argument("--edge-buffer", type=int, default=5)
    parser.add_argument("--timeout-s", type=int, default=8)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--trust-remote-code", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    # Jupyter/Kaggle executes a cell with an internal ``-f <kernel.json>``
    # argument. Accept that one argument pair while keeping normal CLI typos
    # visible to the user.
    args, unknown = parser.parse_known_args()
    if unknown:
        if len(unknown) == 2 and unknown[0] == "-f":
            return args
        parser.error(f"unrecognized arguments: {' '.join(unknown)}")
    return args


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_tasks(task_ids: str, num_tasks: int) -> list[Task]:
    dataset = load_dataset("openai_humaneval", split="test")
    requested_ids = {item.strip() for item in task_ids.split(",") if item.strip()}
    tasks = [
        Task(
            task_id=row["task_id"],
            prompt=row["prompt"],
            test=row["test"],
            entry_point=row["entry_point"],
        )
        for row in dataset
        if not requested_ids or row["task_id"] in requested_ids
    ]
    if requested_ids:
        missing = requested_ids - {task.task_id for task in tasks}
        if missing:
            raise ValueError(f"Unknown HumanEval task IDs: {sorted(missing)}")
        return tasks
    return tasks[:num_tasks]


def build_prompt(task: Task, tokenizer: Any) -> str:
    user_prompt = (
        "Complete the following Python function.\n"
        "Return only valid Python code. Do not use Markdown. Do not explain.\n\n"
        f"{task.prompt}"
    )
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{DEFAULT_SYSTEM_PROMPT}\n\n{user_prompt}"


def strip_markdown_fences(text: str) -> str:
    text = (text or "").strip()
    blocks = re.findall(r"```(?:python|py)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    blocks = [block.strip() for block in blocks if block.strip()]
    if blocks:
        keywords = ("def ", "import ", "from ", "class ", "return ", "assert ")
        return max(blocks, key=lambda block: sum(key in block for key in keywords) * 10 + len(block))
    return text.replace("```python", "").replace("```py", "").replace("```", "").strip()


def extract_code(raw_output: str, entry_point: str) -> str:
    text = strip_markdown_fences(raw_output)
    for marker in ("Explanation:", "Example:", "Examples:", "# Explanation"):
        marker_index = text.find(marker)
        if marker_index != -1:
            text = text[:marker_index].strip()
    match = re.search(rf"def\s+{re.escape(entry_point)}\s*\(", text)
    if match:
        imports = [
            line.strip()
            for line in text[: match.start()].splitlines()
            if line.strip().startswith(("import ", "from "))
        ]
        function_code = text[match.start() :].strip()
        return "\n".join(imports + ([""] if imports else []) + [function_code]).strip()
    return text.strip()


def evaluate(task: Task, raw_output: str, timeout_s: int) -> tuple[bool, str | None]:
    code = extract_code(raw_output, task.entry_point)
    try:
        ast.parse(code)
    except SyntaxError as error:
        return False, f"SyntaxError: {error.msg} at line {error.lineno}"

    prelude = (
        "from typing import *\nimport math\nimport re\nimport itertools\n"
        "import collections\nimport functools\nimport heapq\nimport bisect\n"
        "import string\nimport statistics\nfrom collections import *\n\n"
    )
    source = prelude + code + "\n\n" + task.test + f"\n\ncheck({task.entry_point})\n"
    with tempfile.TemporaryDirectory() as temp_dir:
        candidate_path = Path(temp_dir) / "candidate.py"
        candidate_path.write_text(source, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, str(candidate_path)],
                cwd=temp_dir,
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return False, f"Timeout: exceeded {timeout_s}s"
    if result.returncode == 0:
        return True, None
    stderr = (result.stderr or result.stdout or "unknown execution failure").strip()
    return False, stderr[-800:]


def model_input_device(model: Any) -> torch.device:
    return model.get_input_embeddings().weight.device


def entropy_and_top2(logits: torch.Tensor, tokenizer: Any, position: int) -> TokenTrace:
    logits = logits.float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    entropy = float((-(probabilities * log_probabilities).sum()).item())
    top_values, top_ids = torch.topk(logits, k=2, dim=-1)
    top1_id, top2_id = int(top_ids[0].item()), int(top_ids[1].item())
    return TokenTrace(
        position=position,
        token_id=top1_id,
        token_text=tokenizer.decode([top1_id]),
        entropy=entropy,
        margin=float((top_values[0] - top_values[1]).item()),
        top1_id=top1_id,
        top1_text=tokenizer.decode([top1_id]),
        top2_id=top2_id,
        top2_text=tokenizer.decode([top2_id]),
    )


@torch.inference_mode()
def greedy_generate(model: Any, tokenizer: Any, prompt: str, max_new_tokens: int) -> Generation:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    started = time.perf_counter()
    outputs = model(prompt_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    generated_ids: list[int] = []
    trace: list[TokenTrace] = []
    eos_token_id = tokenizer.eos_token_id

    for position in range(max_new_tokens):
        token_trace = entropy_and_top2(next_logits[0], tokenizer, position)
        trace.append(token_trace)
        next_token_id = token_trace.top1_id
        generated_ids.append(next_token_id)
        if eos_token_id is not None and next_token_id == eos_token_id:
            break
        next_token = torch.tensor([[next_token_id]], device=input_device, dtype=torch.long)
        outputs = model(next_token, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=trace,
        latency_s=time.perf_counter() - started,
    )


@torch.inference_mode()
def generate_top1_top2_batch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    top1_id: int,
    top2_id: int,
    max_new_tokens: int,
) -> tuple[Generation, Generation]:
    """Continue top-1 and top-2 branches in a batch of two equally long prefixes."""

    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=input_device).unsqueeze(0)
    shared_prefix = torch.cat([prompt_ids, prefix_tensor], dim=1) if prefix_ids else prompt_ids
    candidates = torch.tensor([[top1_id], [top2_id]], dtype=torch.long, device=input_device)
    input_ids = torch.cat([shared_prefix.repeat(2, 1), candidates], dim=1)
    started = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    branch_ids = [[top1_id], [top2_id]]
    branch_trace: list[list[TokenTrace]] = [[], []]
    finished = [False, False]
    eos_token_id = tokenizer.eos_token_id
    remaining = max(max_new_tokens - len(prefix_ids) - 1, 0)

    for step in range(remaining):
        next_ids: list[int] = []
        for branch_index in range(2):
            token_trace = entropy_and_top2(next_logits[branch_index], tokenizer, len(prefix_ids) + 1 + step)
            branch_trace[branch_index].append(token_trace)
            token_id = eos_token_id if finished[branch_index] and eos_token_id is not None else token_trace.top1_id
            branch_ids[branch_index].append(int(token_id))
            if eos_token_id is not None and token_id == eos_token_id:
                finished[branch_index] = True
            next_ids.append(int(token_id))
        if all(finished):
            break
        next_tensor = torch.tensor(next_ids, dtype=torch.long, device=input_device).unsqueeze(1)
        outputs = model(next_tensor, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    latency_s = time.perf_counter() - started
    complete_ids = [prefix_ids + branch for branch in branch_ids]
    return (
        Generation(
            token_ids=complete_ids[0],
            code=tokenizer.decode(complete_ids[0], skip_special_tokens=True),
            trace=branch_trace[0],
            latency_s=latency_s,
        ),
        Generation(
            token_ids=complete_ids[1],
            code=tokenizer.decode(complete_ids[1], skip_special_tokens=True),
            trace=branch_trace[1],
            latency_s=latency_s,
        ),
    )


def select_positions(
    trace: list[TokenTrace],
    candidates_per_task: int,
    controls_per_task: int,
    edge_buffer: int,
    rng: np.random.Generator,
) -> list[tuple[str, TokenTrace]]:
    valid = trace[edge_buffer : max(len(trace) - edge_buffer, edge_buffer)]
    high_entropy = sorted(valid, key=lambda item: item.entropy, reverse=True)[:candidates_per_task]
    high_positions = {item.position for item in high_entropy}
    controls_pool = [item for item in valid if item.position not in high_positions]
    controls_count = min(controls_per_task, len(controls_pool))
    controls = (
        [controls_pool[index] for index in rng.choice(len(controls_pool), size=controls_count, replace=False)]
        if controls_count
        else []
    )
    return [("high_entropy", item) for item in high_entropy] + [("random_control", item) for item in controls]


def baseline_record(task: Task, generation: Generation, passed: bool, error: str | None) -> dict[str, Any]:
    return {
        "task_id": task.task_id,
        "passed": passed,
        "error": error,
        "code": generation.code,
        "token_ids": generation.token_ids,
        "trace": [asdict(item) for item in generation.trace],
        "generation_latency_s": generation.latency_s,
    }


def write_summary(branch_records: list[dict[str, Any]], output_dir: Path) -> None:
    rows = []
    for selection_type in ("high_entropy", "random_control"):
        group = [record for record in branch_records if record["selection_type"] == selection_type]
        if not group:
            continue
        recoverable = sum(bool(record["recoverable"]) for record in group)
        rows.append(
            {
                "selection_type": selection_type,
                "n_positions": len(group),
                "recoverable": recoverable,
                "recovery_rate": recoverable / len(group),
                "mean_entropy": float(np.mean([record["entropy"] for record in group])),
                "mean_extra_tokens": float(np.mean([record["extra_tokens"] for record in group])),
            }
        )
    report = {"rows": rows, "created_at_unix": time.time()}
    (output_dir / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    markdown = ["# Top-1 vs Top-2 counterfactual pilot", "", "| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |", "|---|---:|---:|---:|---:|---:|"]
    markdown.extend(
        "| {selection_type} | {n_positions} | {recoverable} | {recovery_rate:.1%} | {mean_entropy:.3f} | {mean_extra_tokens:.1f} |".format(**row)
        for row in rows
    )
    (output_dir / "summary.md").write_text("\n".join(markdown) + "\n", encoding="utf-8")
    print("\n".join(markdown))


def main() -> None:
    args = parse_args()
def run_notebook(
    *,
    task_ids: str,
    model_name: str = DEFAULT_MODEL,
    output_dir: str = "/kaggle/working/top2_counterfactual_pilot",
    max_new_tokens: int = 256,
    candidates_per_task: int = 3,
    controls_per_task: int = 3,
    edge_buffer: int = 5,
    timeout_s: int = 8,
    seed: int = 42,
    trust_remote_code: bool = False,
    overwrite: bool = False,
) -> None:
    """Run the pilot directly from a Kaggle notebook cell.

    Example:
        run_notebook(task_ids="HumanEval/26,HumanEval/38", candidates_per_task=2)
    """

    args = argparse.Namespace(
        model_name=model_name,
        output_dir=output_dir,
        task_ids=task_ids,
        num_tasks=10,
        max_new_tokens=max_new_tokens,
        candidates_per_task=candidates_per_task,
        controls_per_task=controls_per_task,
        edge_buffer=edge_buffer,
        timeout_s=timeout_s,
        seed=seed,
        trust_remote_code=trust_remote_code,
        overwrite=overwrite,
    )
    main(args)


def main(args: argparse.Namespace | None = None) -> None:
    args = args or parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This pilot requires a Kaggle GPU session.")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    rng = np.random.default_rng(args.seed)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    baseline_path = output_dir / "baselines.jsonl"
    branch_path = output_dir / "branches.jsonl"
    metadata_path = output_dir / "metadata.json"
    if args.overwrite:
        for path in (baseline_path, branch_path, metadata_path):
            if path.exists():
                path.unlink()

    tasks = load_tasks(args.task_ids, args.num_tasks)
    metadata_path.write_text(
        json.dumps({"args": vars(args), "tasks": [task.task_id for task in tasks]}, indent=2),
        encoding="utf-8",
    )
    print(f"Loading one model across {torch.cuda.device_count()} GPU(s): {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=args.trust_remote_code)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=args.trust_remote_code,
    )
    model.eval()

    existing_baselines = {record["task_id"]: record for record in read_jsonl(baseline_path)}
    existing_branches = {record["task_id"] for record in read_jsonl(branch_path)}
    for task in tasks:
        if task.task_id not in existing_baselines:
            print(f"Baseline {task.task_id}")
            prompt = build_prompt(task, tokenizer)
            generation = greedy_generate(model, tokenizer, prompt, args.max_new_tokens)
            passed, error = evaluate(task, generation.code, args.timeout_s)
            record = baseline_record(task, generation, passed, error)
            append_jsonl(baseline_path, record)
            existing_baselines[task.task_id] = record
            print(f"  {'PASS' if passed else 'FAIL'} | {len(generation.token_ids)} tokens")

        baseline = existing_baselines[task.task_id]
        if baseline["passed"]:
            print(f"Skip {task.task_id}: baseline passed (pilot targets failures).")
            continue
        if task.task_id in existing_branches:
            print(f"Skip {task.task_id}: branches already saved.")
            continue

        prompt = build_prompt(task, tokenizer)
        trace = [TokenTrace(**item) for item in baseline["trace"]]
        selected = select_positions(
            trace,
            candidates_per_task=args.candidates_per_task,
            controls_per_task=args.controls_per_task,
            edge_buffer=args.edge_buffer,
            rng=rng,
        )
        if not selected:
            print(f"Skip {task.task_id}: too few generated tokens for valid branch positions.")
            continue
        print(f"Branching {task.task_id}: {len(selected)} positions")
        for selection_type, point in selected:
            prefix_ids = baseline["token_ids"][: point.position]
            top1_generation, top2_generation = generate_top1_top2_batch(
                model,
                tokenizer,
                prompt,
                prefix_ids=prefix_ids,
                top1_id=point.top1_id,
                top2_id=point.top2_id,
                max_new_tokens=args.max_new_tokens,
            )
            top1_passed, top1_error = evaluate(task, top1_generation.code, args.timeout_s)
            top2_passed, top2_error = evaluate(task, top2_generation.code, args.timeout_s)
            record = {
                "task_id": task.task_id,
                "selection_type": selection_type,
                "position": point.position,
                "position_relative": point.position / max(len(baseline["token_ids"]) - 1, 1),
                "entropy": point.entropy,
                "margin": point.margin,
                "top1_token": point.top1_text,
                "top2_token": point.top2_text,
                "top1_passed": top1_passed,
                "top1_error": top1_error,
                "top2_passed": top2_passed,
                "top2_error": top2_error,
                "recoverable": (not top1_passed) and top2_passed,
                "top1_matches_baseline": top1_generation.code == baseline["code"],
                "extra_tokens": len(top1_generation.token_ids) + len(top2_generation.token_ids) - 2 * len(prefix_ids),
                "branch_latency_s": top1_generation.latency_s,
                "top1_code": top1_generation.code,
                "top2_code": top2_generation.code,
            }
            append_jsonl(branch_path, record)
            print(
                f"  {selection_type} t={point.position:>3} H={point.entropy:.3f} "
                f"top1={'P' if top1_passed else 'F'} top2={'P' if top2_passed else 'F'}"
            )
        existing_branches.add(task.task_id)
        write_summary(read_jsonl(branch_path), output_dir)

    write_summary(read_jsonl(branch_path), output_dir)
    print(f"\nSaved resumable outputs to: {output_dir}")


if __name__ == "__main__":
    main()

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Skip HumanEval/0: baseline passed (pilot targets failures).
Skip HumanEval/1: baseline passed (pilot targets failures).
Skip HumanEval/2: baseline passed (pilot targets failures).
Skip HumanEval/3: baseline passed (pilot targets failures).
Skip HumanEval/4: baseline passed (pilot targets failures).
Skip HumanEval/5: baseline passed (pilot targets failures).
Skip HumanEval/6: baseline passed (pilot targets failures).
Skip HumanEval/7: baseline passed (pilot targets failures).
Skip HumanEval/8: baseline passed (pilot targets failures).
Skip HumanEval/9: baseline passed (pilot targets failures).
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [2]:
# ============================================================
# REVISIÓN: SEMANTIC LOOKAHEAD CON ALINEACIÓN DE TOKENS
# ============================================================

import ast
import csv
import gc
import json
import math
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

ALIGNED_OUTPUT_DIR = Path(
    "/kaggle/working/aligned_semantic_lookahead_validation"
)
ALIGNED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALIGNED_SELECTIONS_PATH = ALIGNED_OUTPUT_DIR / "aligned_selections.jsonl"
ALIGNED_BRANCHES_PATH = ALIGNED_OUTPUT_DIR / "aligned_branches.jsonl"
ALIGNED_POSITIONS_PATH = ALIGNED_OUTPUT_DIR / "aligned_positions.csv"
ALIGNED_REPORT_PATH = ALIGNED_OUTPUT_DIR / "aligned_report.json"
ALIGNED_ZIP_PATH = Path(
    "/kaggle/working/aligned_semantic_lookahead_validation.zip"
)

BRANCH_BUDGET_ALIGNED = 5


# ------------------------------------------------------------
# Localizar o extraer el checkpoint anterior
# ------------------------------------------------------------

def locate_checkpoint():
    direct = Path("/kaggle/working/frozen_selector_validation")

    required = {
        "baselines.jsonl",
        "lookaheads.jsonl",
        "branches.jsonl",
        "selections.jsonl",
        "validation_report.json",
    }

    if direct.is_dir():
        names = {path.name for path in direct.iterdir()}
        if required.issubset(names):
            return direct

    zip_candidates = []

    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            zip_candidates.extend(
                root.rglob("frozen_selector_validation_checkpoint.zip")
            )

    if not zip_candidates:
        raise FileNotFoundError(
            "No encontré frozen_selector_validation_checkpoint.zip. "
            "Subilo como Kaggle Input o ejecutá primero la validación anterior."
        )

    extract_dir = Path(
        "/kaggle/working/frozen_selector_validation"
    )
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(extract_dir)

    return extract_dir


SOURCE_DIR = locate_checkpoint()

BASELINES_SOURCE = SOURCE_DIR / "baselines.jsonl"
LOOKAHEADS_SOURCE = SOURCE_DIR / "lookaheads.jsonl"
BRANCHES_SOURCE = SOURCE_DIR / "branches.jsonl"
SELECTIONS_SOURCE = SOURCE_DIR / "selections.jsonl"
REPORT_SOURCE = SOURCE_DIR / "validation_report.json"


def read_jsonl(path):
    if not path.exists():
        return []

    with path.open("r", encoding="utf-8") as handle:
        return [
            json.loads(line)
            for line in handle
            if line.strip()
        ]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )


# ------------------------------------------------------------
# Alineación Levenshtein con reconstrucción de pares
# ------------------------------------------------------------

def levenshtein_alignment(left, right):
    """
    Devuelve:
      - distancia mínima;
      - pares (índice_left, índice_right).

    None representa una inserción o eliminación.
    """

    n = len(left)
    m = len(right)

    matrix = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(n + 1):
        matrix[i][0] = i

    for j in range(m + 1):
        matrix[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            substitution = (
                0 if left[i - 1] == right[j - 1] else 1
            )

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,
                matrix[i][j - 1] + 1,
                matrix[i - 1][j - 1] + substitution,
            )

    alignment = []
    i = n
    j = m

    while i > 0 or j > 0:
        if (
            i > 0
            and j > 0
            and matrix[i][j]
            == matrix[i - 1][j - 1]
            + (left[i - 1] != right[j - 1])
        ):
            alignment.append((i - 1, j - 1))
            i -= 1
            j -= 1

        elif (
            i > 0
            and matrix[i][j] == matrix[i - 1][j] + 1
        ):
            alignment.append((i - 1, None))
            i -= 1

        else:
            alignment.append((None, j - 1))
            j -= 1

    alignment.reverse()

    return matrix[n][m], alignment


# ------------------------------------------------------------
# Peso léxico de una diferencia alineada
# ------------------------------------------------------------

def aligned_difference_weight(left_text, right_text):
    if left_text == right_text:
        return 0.0

    left_class = token_class(left_text)
    right_class = token_class(right_text)
    classes = {left_class, right_class}

    if classes == {"whitespace"}:
        return 0.0

    if classes == {"identifier"}:
        return 0.20

    if "whitespace" in classes:
        return 0.25

    if classes & {"keyword", "operator", "literal"}:
        return 1.0

    return 0.50


def calculate_aligned_features(
    lookahead,
    baseline,
):
    """
    Excluye el token forzado y alinea solamente la continuación.
    """

    baseline_ids = lookahead["baseline_token_ids"][1:]
    branch_ids = lookahead["branch_token_ids"][1:]

    distance, alignment = levenshtein_alignment(
        baseline_ids,
        branch_ids,
    )

    denominator = max(
        len(baseline_ids),
        len(branch_ids),
        1,
    )

    aligned_persistence = distance / denominator

    position = int(lookahead["position"])
    trace = baseline["trace"]

    # Textos correspondientes a los tokens posteriores
    # al token top-1 original.
    baseline_texts = [
        trace[position + index]["token_text"]
        for index in range(1, len(lookahead["baseline_token_ids"]))
    ]

    branch_texts = lookahead["branch_token_texts"][1:]

    weighted_total = 0.0

    for left_index, right_index in alignment:
        left_text = (
            baseline_texts[left_index]
            if left_index is not None
            else ""
        )

        right_text = (
            branch_texts[right_index]
            if right_index is not None
            else ""
        )

        weighted_total += aligned_difference_weight(
            left_text,
            right_text,
        )

    aligned_weighted_divergence = (
        weighted_total / max(len(alignment), 1)
    )

    positional_mismatches = sum(
        left != right
        for left, right in zip(baseline_ids, branch_ids)
    ) + abs(len(baseline_ids) - len(branch_ids))

    return {
        "aligned_edit_distance": int(distance),
        "aligned_persistence": float(aligned_persistence),
        "aligned_weighted_divergence": float(
            aligned_weighted_divergence
        ),
        "alignment_operations": len(alignment),
        "positional_mismatches": int(positional_mismatches),
        "shift_detected": bool(
            distance < positional_mismatches
        ),
    }


# ------------------------------------------------------------
# Percentiles con empates
# ------------------------------------------------------------

def aligned_midrank_percentiles(rows, field):
    groups = defaultdict(list)

    for row in rows:
        groups[float(row[field])].append(row)

    ordered_values = sorted(groups, reverse=True)
    denominator = max(len(rows) - 1, 1)

    cursor = 1
    scores = {}

    for value in ordered_values:
        group = groups[value]

        mean_rank = cursor + (len(group) - 1) / 2
        percentile = (
            1.0 - (mean_rank - 1) / denominator
        )

        for row in group:
            scores[int(row["position"])] = percentile

        cursor += len(group)

    return scores


def add_aligned_semantic_scores(rows):
    components = (
        "aligned_persistence",
        "aligned_weighted_divergence",
        "anchor_score",
    )

    percentiles = {
        field: aligned_midrank_percentiles(rows, field)
        for field in components
    }

    for row in rows:
        position = int(row["position"])

        row["aligned_semantic_score"] = float(
            np.mean([
                percentiles[field][position]
                for field in components
            ])
        )

    return rows


# ------------------------------------------------------------
# Cargar datos y recalcular señales
# ------------------------------------------------------------

baselines = {
    row["task_id"]: row
    for row in read_jsonl(BASELINES_SOURCE)
}

lookaheads = read_jsonl(LOOKAHEADS_SOURCE)

previous_branches = {
    (row["task_id"], int(row["position"])): row
    for row in read_jsonl(BRANCHES_SOURCE)
}

previous_selections = {
    row["task_id"]: row
    for row in read_jsonl(SELECTIONS_SOURCE)
}

with REPORT_SOURCE.open("r", encoding="utf-8") as handle:
    previous_report = json.load(handle)

failure_task_ids = previous_report[
    "baseline_failures_selected"
]

lookaheads_by_task = defaultdict(list)

for lookahead in lookaheads:
    task_id = lookahead["task_id"]

    aligned_features = calculate_aligned_features(
        lookahead,
        baselines[task_id],
    )

    revised = {
        **lookahead,
        **aligned_features,
    }

    lookaheads_by_task[task_id].append(revised)


aligned_selections = {}
all_aligned_rows = []

for task_id in failure_task_ids:
    rows = add_aligned_semantic_scores(
        lookaheads_by_task[task_id]
    )

    ranked = sorted(
        rows,
        key=lambda row: (
            -row["aligned_semantic_score"],
            deterministic_tie_break(
                task_id,
                "aligned_semantic",
                int(row["position"]),
            ),
        ),
    )

    selected_positions = [
        int(row["position"])
        for row in ranked[:BRANCH_BUDGET_ALIGNED]
    ]

    old_positions = previous_selections[
        task_id
    ]["by_selector"]["semantic_lookahead"]

    aligned_selections[task_id] = {
        "task_id": task_id,
        "aligned_semantic_positions": selected_positions,
        "old_semantic_positions": old_positions,
        "overlap": len(
            set(selected_positions) & set(old_positions)
        ),
        "eligible_positions": len(rows),
    }

    selected_set = set(selected_positions)

    for row in rows:
        row["selected_by_aligned_semantic"] = (
            int(row["position"]) in selected_set
        )
        all_aligned_rows.append(row)


# Guardar selecciones recalculadas
ALIGNED_SELECTIONS_PATH.write_text(
    "".join(
        json.dumps(record, ensure_ascii=False) + "\n"
        for record in aligned_selections.values()
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# Determinar qué ramas ya existen y cuáles faltan
# ------------------------------------------------------------

aligned_branch_results = {
    (row["task_id"], int(row["position"])): row
    for row in read_jsonl(ALIGNED_BRANCHES_PATH)
}

missing = []

for task_id, selection in aligned_selections.items():
    for position in selection["aligned_semantic_positions"]:
        key = (task_id, position)

        if key in aligned_branch_results:
            continue

        if key in previous_branches:
            old = previous_branches[key]

            reused = {
                **old,
                "selected_by": ["aligned_semantic"],
                "reused_from_previous_validation": True,
            }

            append_jsonl(
                ALIGNED_BRANCHES_PATH,
                reused,
            )
            aligned_branch_results[key] = reused

        else:
            missing.append(key)


print(
    f"Ramas seleccionadas: "
    f"{len(failure_task_ids) * BRANCH_BUDGET_ALIGNED}"
)
print(
    f"Ramas reutilizadas o ya completas: "
    f"{len(aligned_branch_results)}"
)
print(f"Ramas nuevas que faltan: {len(missing)}")


# ------------------------------------------------------------
# Generar solamente las ramas faltantes
# ------------------------------------------------------------

if missing:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Activá una GPU en Kaggle para generar "
            "las ramas faltantes."
        )

    all_tasks = load_tasks("", num_tasks=164)
    tasks_by_id = {
        task.task_id: task
        for task in all_tasks
    }

    gc.collect()
    torch.cuda.empty_cache()

    print(f"Cargando {VALIDATION_MODEL}...")

    tokenizer = AutoTokenizer.from_pretrained(
        VALIDATION_MODEL
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        VALIDATION_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    rows_by_key = {
        (row["task_id"], int(row["position"])): row
        for row in all_aligned_rows
    }

    for index, (task_id, position) in enumerate(
        missing,
        start=1,
    ):
        task = tasks_by_id[task_id]
        baseline = baselines[task_id]
        lookahead = rows_by_key[(task_id, position)]

        prompt = build_prompt(task, tokenizer)

        generated = generate_full_forced_branch(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            prefix_ids=baseline["token_ids"][:position],
            forced_token_id=int(
                lookahead["top2_token_id"]
            ),
        )

        passed, error = evaluate(
            task,
            generated["code"],
            TEST_TIMEOUT_S,
        )

        record = {
            "task_id": task_id,
            "position": position,
            "selected_by": ["aligned_semantic"],
            "passed": bool(passed),
            "error": error,
            "top1_token": lookahead["top1_token"],
            "top2_token": lookahead["top2_token"],
            "top1_token_id": lookahead["top1_token_id"],
            "top2_token_id": lookahead["top2_token_id"],
            "aligned_semantic_score": lookahead[
                "aligned_semantic_score"
            ],
            "aligned_persistence": lookahead[
                "aligned_persistence"
            ],
            "aligned_weighted_divergence": lookahead[
                "aligned_weighted_divergence"
            ],
            "anchor_score": lookahead["anchor_score"],
            "shift_detected": lookahead["shift_detected"],
            "reused_from_previous_validation": False,
            **generated,
        }

        append_jsonl(ALIGNED_BRANCHES_PATH, record)
        aligned_branch_results[(task_id, position)] = record

        print(
            f"[{index}/{len(missing)}] "
            f"{task_id} t={position}: "
            f"{'PASS' if passed else 'FAIL'}"
        )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Resultado final
# ------------------------------------------------------------

task_success = {}

for task_id, selection in aligned_selections.items():
    task_success[task_id] = any(
        aligned_branch_results[
            (task_id, position)
        ]["passed"]
        for position in selection[
            "aligned_semantic_positions"
        ]
    )

recovered_tasks = sum(task_success.values())
total_tasks = len(task_success)
recovery_rate = recovered_tasks / total_tasks

# Intervalo de Wilson 95 %
z = 1.959963984540054
denominator = 1 + z**2 / total_tasks
center = (
    recovery_rate
    + z**2 / (2 * total_tasks)
) / denominator
half_width = (
    z
    * math.sqrt(
        recovery_rate * (1 - recovery_rate) / total_tasks
        + z**2 / (4 * total_tasks**2)
    )
    / denominator
)

shift_rows = [
    row for row in all_aligned_rows
    if row["shift_detected"]
]

selected_rows = [
    row for row in all_aligned_rows
    if row["selected_by_aligned_semantic"]
]

selected_shift_rows = [
    row for row in selected_rows
    if row["shift_detected"]
]

report = {
    "status": "complete",
    "model": VALIDATION_MODEL,
    "tasks": total_tasks,
    "branch_budget_per_task": BRANCH_BUDGET_ALIGNED,
    "lookahead_tokens_total": LOOKAHEAD_TOKENS,
    "continuation_tokens_aligned": LOOKAHEAD_TOKENS - 1,
    "alignment": (
        "unit-cost Levenshtein over continuation tokens; "
        "forced token excluded"
    ),
    "semantic_components": [
        "aligned_persistence",
        "aligned_weighted_divergence",
        "anchor_score",
    ],
    "recovered_tasks": recovered_tasks,
    "recovery_rate": recovery_rate,
    "wilson_95_interval": [
        center - half_width,
        center + half_width,
    ],
    "total_lookahead_positions": len(all_aligned_rows),
    "shift_affected_positions": len(shift_rows),
    "shift_affected_rate": (
        len(shift_rows) / len(all_aligned_rows)
    ),
    "selected_shift_affected_positions": len(
        selected_shift_rows
    ),
    "mean_top5_overlap_with_old_selector": float(
        np.mean([
            selection["overlap"]
            for selection in aligned_selections.values()
        ])
    ),
    "task_results": task_success,
}

ALIGNED_REPORT_PATH.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


# CSV de posiciones y métricas
csv_fields = [
    "task_id",
    "position",
    "top1_token",
    "top2_token",
    "aligned_persistence",
    "aligned_weighted_divergence",
    "anchor_score",
    "aligned_semantic_score",
    "shift_detected",
    "selected_by_aligned_semantic",
    "persistence_after_forced",
    "weighted_semantic_divergence",
    "normalized_edit_distance",
    "baseline_snippet",
    "branch_snippet",
]

with ALIGNED_POSITIONS_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=csv_fields,
    )
    writer.writeheader()

    for row in sorted(
        all_aligned_rows,
        key=lambda item: (
            item["task_id"],
            int(item["position"]),
        ),
    ):
        writer.writerow({
            field: row.get(field)
            for field in csv_fields
        })


# ZIP descargable
if ALIGNED_ZIP_PATH.exists():
    ALIGNED_ZIP_PATH.unlink()

with zipfile.ZipFile(
    ALIGNED_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in ALIGNED_OUTPUT_DIR.iterdir():
        if path.is_file():
            archive.write(path, arcname=path.name)


print("\nRESULTADO FINAL")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nDescargá este archivo:")
print(ALIGNED_ZIP_PATH)

Ramas seleccionadas: 100
Ramas reutilizadas o ya completas: 100
Ramas nuevas que faltan: 0

RESULTADO FINAL
{
  "status": "complete",
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "tasks": 20,
  "branch_budget_per_task": 5,
  "lookahead_tokens_total": 12,
  "continuation_tokens_aligned": 11,
  "alignment": "unit-cost Levenshtein over continuation tokens; forced token excluded",
  "semantic_components": [
    "aligned_persistence",
    "aligned_weighted_divergence",
    "anchor_score"
  ],
  "recovered_tasks": 10,
  "recovery_rate": 0.5,
  "wilson_95_interval": [
    0.2992980081982123,
    0.7007019918017877
  ],
  "total_lookahead_positions": 1649,
  "shift_affected_positions": 679,
  "shift_affected_rate": 0.4117647058823529,
  "selected_shift_affected_positions": 15,
  "mean_top5_overlap_with_old_selector": 3.35,
  "task_results": {
    "HumanEval/140": true,
    "HumanEval/93": false,
    "HumanEval/10": false,
    "HumanEval/77": true,
    "HumanEval/141": true,
    "HumanEval/15